In [1]:
import os
import random
import sys
import cv2
import tqdm
import json
from PIL import Image
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import multilabel_confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
!pip install polars seaborn

import polars as pl

In [3]:
!nvidia-smi

Wed Apr 29 10:48:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               On  |   00000000:56:00.0 Off |                  Off |
| 38%   65C    P2            178W /  230W |    9143MiB /  24564MiB |     93%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import os
os.chdir('/workspace')

In [5]:
df = pl.read_csv("vindr_mammogram/mammo_processed_merged1/mammo_processed_merged.csv")
df = df.to_pandas()
df["merged_image_path"] = (
    df["merged_image_path"]
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
    .str.replace("vindr_original_data", "vindr_mammogram", regex=False)
)

In [6]:
df['cc_finding_categories'].value_counts().sort_index()

cc_finding_categories
['Architectural Distortion', 'Mass']                                                                   1
['Architectural Distortion']                                                                          40
['Asymmetry', 'Mass']                                                                                  1
['Asymmetry']                                                                                         20
['Focal Asymmetry']                                                                                  107
['Global Asymmetry']                                                                                  11
['Mass']                                                                                             443
['Nipple Retraction', 'Mass']                                                                          1
['Nipple Retraction', 'Skin Thickening', 'Mass']                                                       1
['Nipple Retraction']            

In [7]:
def get_combined_finding_6class(cc_findings, mlo_findings, cc_birads, mlo_birads):
    if isinstance(cc_findings, str):
        cc_findings = eval(cc_findings) if cc_findings else []
    if isinstance(mlo_findings, str):
        mlo_findings = eval(mlo_findings) if mlo_findings else []
    
    cc_findings = cc_findings or []
    mlo_findings = mlo_findings or []
    all_findings = set(cc_findings + mlo_findings)
    
    if len(all_findings) > 1 and 'No Finding' in all_findings:
        all_findings.remove('No Finding')
    
    high_suspicion_structural = {
        'Architectural Distortion',
        'Skin Thickening',
        'Skin Retraction',
        'Nipple Retraction'
    }
    
    asymmetry_findings = {
        'Focal Asymmetry',
        'Global Asymmetry',
        'Asymmetry'
    }
    
    has_mass = 'Mass' in all_findings
    has_calc = 'Suspicious Calcification' in all_findings
    has_structural = bool(all_findings & high_suspicion_structural)
    has_asymmetry = bool(all_findings & asymmetry_findings)
    has_lymph = 'Suspicious Lymph Node' in all_findings
    
    def parse_birads(birads_str):
        if pd.isna(birads_str) or birads_str == '':
            return 0
        if isinstance(birads_str, str):
            try:
                return int(birads_str.strip().split()[-1])
            except:
                return 0
        return int(birads_str)
    
    cc_birads_num = parse_birads(cc_birads)
    mlo_birads_num = parse_birads(mlo_birads)
    max_birads = max(cc_birads_num, mlo_birads_num)
    
    if not all_findings or all_findings == {'No Finding'}:
        if max_birads == 1:
            return 0
        elif max_birads == 2:
            return 1
        else:
            return 1 if max_birads == 3 else 4
    
    if (has_mass and has_calc) or has_lymph:
        return 4  # Complex/Combined
    
    # Priority 5: Architectural distortions (without mass)
    if has_structural:
        return 5
    
    # Priority 3: Mass (isolated)
    if has_mass:
        return 3
    
    # Priority 2: Calcification (isolated)
    if has_calc:
        return 2
    
    if has_asymmetry and len(all_findings) == 1:
        return -1
    
    if has_asymmetry and len(all_findings) > 1:
        return 5
    
    print(f"Warning: Unknown finding combination: {all_findings}, BIRADS: {max_birads}")
    return 5

df['finding'] = df.apply(
    lambda row: get_combined_finding_6class(
        row['cc_finding_categories'], 
        row['mlo_finding_categories'],
        row['cc_breast_birads'],
        row['mlo_breast_birads']
    ),
    axis=1
)
df.drop(df[df['finding'] == -1].index, inplace=True)
df['finding'].value_counts().sort_index()

finding
0    6703
1    2329
2     127
3     483
4      85
5      83
Name: count, dtype: int64

In [8]:
import ast
import pandas as pd
from collections import Counter

ASYMMETRY_FINDINGS  = frozenset({"Asymmetry", "Focal Asymmetry", "Global Asymmetry"})
STRUCTURAL_FINDINGS = frozenset({"Architectural Distortion", "Skin Thickening", "Skin Retraction", "Nipple Retraction"})
FINDING_TO_BINARY   = [0, 1, 1, 1, 1, 1]
NUM_PROTOTYPES      = 6

def _parse_findings(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return frozenset()
    if isinstance(raw, (list, tuple, set)):
        return frozenset(str(f).strip() for f in raw if str(f).strip())
    if isinstance(raw, str):
        raw = raw.strip()
        if not raw:
            return frozenset()
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, (list, tuple, set)):
                return frozenset(str(f).strip() for f in parsed if str(f).strip())
        except (ValueError, SyntaxError):
            pass
        return frozenset({raw})
    return frozenset()

def _parse_birads(raw):
    if raw is None or (isinstance(raw, float) and pd.isna(raw)):
        return 0
    if isinstance(raw, (int, float)):
        return int(raw)
    s = str(raw).strip()
    for token in reversed(s.split()):
        digits = "".join(c for c in token if c.isdigit())
        if digits:
            return int(digits[0])
    return 0

def add_finding_columns(df,
                        cc_findings_col="cc_finding_categories",
                        mlo_findings_col="mlo_finding_categories",
                        cc_birads_col="cc_breast_birads",
                        mlo_birads_col="mlo_breast_birads"):
    rows, drop_idx, other_log = [], [], []

    for idx, row in df.iterrows():
        cc       = _parse_findings(row[cc_findings_col])
        mlo      = _parse_findings(row[mlo_findings_col])
        combined = cc | mlo

        if len(combined) > 1 and "No Finding" in combined:
            combined = combined - {"No Finding"}

        non_asym = combined - ASYMMETRY_FINDINGS
        if combined and not non_asym:
            drop_idx.append(idx)
            continue
        combined = non_asym

        max_birads  = max(_parse_birads(row[cc_birads_col]), _parse_birads(row[mlo_birads_col]))
        is_negative = not combined or combined == {"No Finding"}

        KNOWN = {"No Finding", "Mass", "Suspicious Calcification", "Suspicious Lymph Node"}
        remaining = combined - KNOWN - STRUCTURAL_FINDINGS

        if not is_negative and remaining:
            other_log.extend(list(remaining))

        rows.append({
            "idx":                idx,
            "has_neg_b1":         int(is_negative and max_birads <= 1),
            "has_neg_b2":         int(is_negative and max_birads > 1),
            "has_mass":           int("Mass" in combined),
            "has_calc":           int("Suspicious Calcification" in combined),
            "has_structural":     int(bool(combined & STRUCTURAL_FINDINGS)),
            "has_lymph": int(not is_negative and (
                                      "Suspicious Lymph Node" in combined or bool(remaining))),
        })

    print("=== Top findings landing in has_lymph_or_other ===")
    for finding, count in Counter(other_log).most_common(20):
        print(f"  {finding}: {count}")

    encoded = pd.DataFrame(rows).set_index("idx")
    df = df.drop(index=drop_idx).join(encoded).reset_index(drop=True)
    return df

df = add_finding_columns(df)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph"]
print("\n=== Finding counts ===")
print(df[FINDING_COLS].sum())


=== Top findings landing in has_lymph_or_other ===

=== Finding counts ===
has_neg_b1        6703
has_neg_b2        2329
has_mass           570
has_calc           207
has_structural      90
has_lymph           22
dtype: int64


In [9]:
inbreast_df = pd.read_csv("inbreast_data/INbreast_merged/merged_metadata.csv")
inbreast_df["merged_image_path"] = (
    inbreast_df["merged_image_path"]
    .str.replace("INbreast Release 1.0", "inbreast_data", regex=False)
    .str.replace("vindr_original_data", "inbreast_data", regex=False))
inbreast_df['birads'] = inbreast_df['birads'].replace({'4a': '4', '4b': '4', '4c': '4','6':'5'})
inbreast_df['label'] = (inbreast_df['birads'].astype(int) - 1).astype(int)
inbreast_df['density'] = 0
inbreast_df['finding'] = 0
inbreast_df['cc_breast_birads'] = None
inbreast_df['mlo_breast_birads'] = None
inbreast_df['cc_breast_density'] = None
inbreast_df['device_id'] = 0
for col in ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]:
    inbreast_df[col] = 0
inbreast_df.head()

,patient_id,laterality,merged_image_path,cc_file_name,mlo_file_name,cc_image_path,mlo_image_path,birads,cc_roi_width,cc_roi_height,...,mlo_breast_birads,cc_breast_density,device_id,has_neg_b1,has_neg_b2,has_mass,has_calc,has_structural,has_lymph,has_other
0,024ee3569b2605dc,L,inbreast_data/INbreast_merged/024ee3569b2605dc...,20588020,20588072,INbreast Release 1.0/INbreast_processed/205880...,INbreast Release 1.0/INbreast_processed/205880...,2,1557,3231,...,None,None,0,0,0,0,0,0,0,0
1,024ee3569b2605dc,R,inbreast_data/INbreast_merged/024ee3569b2605dc...,20587994,20588046,INbreast Release 1.0/INbreast_processed/205879...,INbreast Release 1.0/INbreast_processed/205880...,5,1535,3128,...,None,None,0,0,0,0,0,0,0,0
2,069212ec65a94339,L,inbreast_data/INbreast_merged/069212ec65a94339...,50994787,50994733,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1226,2580,...,None,None,0,0,0,0,0,0,0,0
3,069212ec65a94339,R,inbreast_data/INbreast_merged/069212ec65a94339...,50994706,50994760,INbreast Release 1.0/INbreast_processed/509947...,INbreast Release 1.0/INbreast_processed/509947...,1,1128,2566,...,None,None,0,0,0,0,0,0,0,0
4,0b7396cdccacca82,L,inbreast_data/INbreast_merged/0b7396cdccacca82...,22670832,22670878,INbreast Release 1.0/INbreast_processed/226708...,INbreast Release 1.0/INbreast_processed/226708...,2,1627,2983,...,None,None,0,0,0,0,0,0,0,0


In [10]:
inbreast_df['label'].value_counts()

label
1    98
0    30
4    28
3    20
2    11
Name: count, dtype: int64

In [11]:
def birads_to_label(birads_category):
    """Convert BI-RADS categories to numerical labels 0-4 (for 5 classes)"""
    birads_num = int(birads_category.replace(" ", "")[-1])
    return birads_num - 1
df['label'] = df['cc_breast_birads'].apply(birads_to_label)

In [12]:
df['label'].value_counts()

label
0    6703
1    2337
3     339
2     319
4     112
Name: count, dtype: int64

In [13]:
df['cc_breast_density'].value_counts()

cc_breast_density
DENSITY C    7486
DENSITY D    1335
DENSITY B     942
DENSITY A      47
Name: count, dtype: int64

In [14]:
data = df[df['split'] == 'training']
test_df = df[df['split'] == 'test']

In [15]:
from sklearn.model_selection import train_test_split


study_meta = (
    data
    .groupby('study_id')
    .agg({
        'cc_breast_birads': 'first',   # BI-RADS at study level
        'finding': 'first'             # finding already encoded as 0–4
    })
    .reset_index()
)


# -------------------------------------------------
study_meta['stratify_key'] = (
    study_meta['cc_breast_birads'].astype(str) + '_' +
    study_meta['finding'].astype(str)
)


train_studies, val_studies = train_test_split(
    study_meta['study_id'],
    test_size=0.1,
    stratify=study_meta['stratify_key'],
    random_state=423
)

train_df = data[data['study_id'].isin(train_studies)].copy()
val_df   = data[data['study_id'].isin(val_studies)].copy()


In [16]:
train_df.shape

(7057, 32)

In [17]:
import numpy as np
import cv2
from PIL import Image
import torchvision.transforms as transforms
import random
import torch


def get_transforms(img_size=(512, 512)):
    """Enhanced mammogram preprocessing with medical imaging considerations"""
    
    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        
        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=10,
                translate=(0.1, 0.1),
                scale=(0.9, 1.1),
                shear=6
            )
        ], p=0.6),
        
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.RandomPerspective(distortion_scale=0.1, p=1.0)
        ], p=0.1),
        
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=5.0, sigma=3.0)
        ], p=0.2),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.95, 1.05),
                contrast=(0.9, 1.1)
            )
        ], p=0.4),
        transforms.Lambda(lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2)) 
                         if random.random() < 0.4 else x),
        

        
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0))
        ], p=0.2),
        
                # NOISE AUGMENTATIONS
        transforms.Lambda(lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.02)) 
                         if random.random() < 0.4 else x),
        
        transforms.Lambda(lambda x: add_speckle_noise(x, std=random.uniform(0.01, 0.03)) 
                         if random.random() < 0.2 else x),
        transforms.ToTensor(),
        
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return train_transform, val_transform

def add_gaussian_noise(image, mean=0, std=0.02):
    """Gaussian noise - electronic noise in imaging sensors"""
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(mean, std, img_array.shape)
        noisy_img = np.clip(img_array + noise, 0, 1)
        return Image.fromarray((noisy_img * 255).astype(np.uint8))
    return image


def adjust_gamma(image, gamma=1.0):
    """
    Gamma correction - handles tissue density variation
    Gamma < 1 = brighter, > 1 = darker
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image).astype(np.float32) / 255.0
        gamma_corrected = np.power(img_array, gamma)
        return Image.fromarray((gamma_corrected * 255).astype(np.uint8))
    return image

In [18]:
#### import numpy as np
import random
from PIL import Image
import torchvision.transforms as T

# ── Noise helpers (unchanged — domain-correct) ────────────────────────────────
def add_gaussian_noise(image, mean=0, std=0.02):
    """Electronic sensor noise — additive Gaussian"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noisy = np.clip(arr + np.random.normal(mean, std, arr.shape), 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def add_speckle_noise(image, std=0.03):
    """Multiplicative speckle — physically correct for mammography"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        noise = np.random.normal(0, std, arr.shape)
        noisy = np.clip(arr + arr * noise, 0, 1)
        return Image.fromarray((noisy * 255).astype(np.uint8))
    return image

def adjust_gamma(image, gamma=1.0):
    """Gamma correction — tissue density variation simulation"""
    if isinstance(image, Image.Image):
        arr = np.array(image).astype(np.float32) / 255.0
        corrected = np.power(arr, gamma)
        return Image.fromarray((corrected * 255).astype(np.uint8))
    return image

def random_90_rotation(image):
    """
    Only 90°/180°/270° steps — avoids interpolation artifact on
    microcalcifications (Shia et al. 2024, Hamidinekoo et al. 2017)
    """
    k = random.choice([0, 1, 2, 3])
    return image.rotate(k * 90)

def get_transforms(img_size=(512, 512)):

    train_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),

        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.RandomPerspective(distortion_scale=0.1, p=0.2),
        transforms.RandomApply([
            transforms.ElasticTransform(alpha=15.0, sigma=4.0)
        ], p=0.25),

        transforms.RandomApply([
            transforms.RandomAffine(
                degrees=5,
                translate=None,
                scale=(0.9, 1.05),
                shear=0,
            )
        ], p=0.5),

        # Contrast + brightness — tissue density varies across patients
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=(0.8, 1.2),
                contrast=(0.7, 1.3),
            )
        ], p=0.5),

        # Gamma — handles exposure variation between machines
        transforms.Lambda(
            lambda x: adjust_gamma(x, gamma=random.uniform(0.8, 1.2))
            if random.random() < 0.4 else x
        ),

        # Gaussian noise — simulates sensor noise, keep it very mild
        transforms.Lambda(
            lambda x: add_gaussian_noise(x, mean=0, std=random.uniform(0.001, 0.03))
            if random.random() < 0.3 else x
        ),

        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    val_transform = transforms.Compose([
        transforms.Resize(img_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ])

    return train_transform, val_transform

In [19]:
import os
import torch
import random
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

Image.MAX_IMAGE_PIXELS = None
torch.manual_seed(2024)

FINDING_COLS = ["has_neg_b1", "has_neg_b2", "has_mass", "has_calc", "has_structural", "has_lymph", "has_other"]


class CustomDataset(Dataset):
    def __init__(self, df, transformations=None, ref_transform=None):
        self.df            = df.reset_index(drop=True)
        self.transform     = transformations
        self.ref_transform = ref_transform
        self.cls_names     = {0: "birads_0", 1: "birads_1",2: "birads_2",3: "birads_3",4: "birads_4"}
        self.cls_counts    = df['label'].value_counts().to_dict()

    def __len__(self):
        return len(self.df)

    def get_cls_name(self, label):
        return self.cls_names[label]


    def __getitem__(self, idx):
        row         = self.df.iloc[idx]
        qry_label   = row['label']
        qry_density = row['cc_breast_density']
        qry_finding = row['finding']
        # qry_device  = row['device_id']

        qry_im = Image.open(row['merged_image_path']).convert("RGB")

        if self.transform:
            qry_im = self.transform(qry_im)

        finding_vec = torch.tensor(
            [row[col] for col in FINDING_COLS], dtype=torch.float32
        )

        return {
            "qry_im":      qry_im,
            "qry_gt":      torch.tensor(qry_label,   dtype=torch.long),
            "finding":     qry_finding,
            "finding_vec": finding_vec,
            "has_neg_b1":     torch.tensor(row["has_neg_b1"],     dtype=torch.long),
            "has_neg_b2":     torch.tensor(row["has_neg_b2"],     dtype=torch.long),
            "has_mass":       torch.tensor(row["has_mass"],       dtype=torch.long),
            "has_calc":       torch.tensor(row["has_calc"],       dtype=torch.long),
            "has_structural": torch.tensor(row["has_structural"], dtype=torch.long),
            "has_lymph":      torch.tensor(row["has_lymph"],      dtype=torch.long)
        }


def get_dls(train_df, val_df, test_df, inbreast_df, bs, ns=6):
    train_trans, val_trans = get_transforms(img_size=(512, 512))

    tr_ds       = CustomDataset(train_df,    train_trans, val_trans)
    vl_ds       = CustomDataset(val_df,      val_trans,   val_trans)
    test_ds     = CustomDataset(test_df,     val_trans,   val_trans)
    inbreast_ds = CustomDataset(inbreast_df, val_trans,   val_trans)

    labels                       = train_df['label'].values
    unique_classes, class_counts = np.unique(labels, return_counts=True)
    beta                         = 0.5
    class_weights                = (1.0 / class_counts) ** beta
    class_weights                = class_weights / class_weights.sum() * len(unique_classes)
    sample_weights               = class_weights[labels]

    print("Class counts:",           dict(zip(unique_classes, class_counts)))
    print("Smoothed class weights:", np.round(class_weights, 3))

    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(sample_weights),
        replacement = True,
    )

    tr_dl = DataLoader(
        tr_ds, batch_size=bs, 
        shuffle=True,
        # sampler=sampler,
        drop_last=True, num_workers=4, pin_memory=True, persistent_workers=True,
    )
    val_dl = DataLoader(
        vl_ds, batch_size=bs, shuffle=False,
        drop_last=False, num_workers=8, pin_memory=True, persistent_workers=True,
    )
    ts_dl       = DataLoader(test_ds,     batch_size=1, shuffle=False, num_workers=ns)
    inbreast_dl = DataLoader(inbreast_ds, batch_size=1, shuffle=False, num_workers=ns)

    return tr_dl, val_dl, ts_dl, inbreast_dl, tr_ds.cls_names, tr_ds.cls_counts

In [20]:
tr_dl, val_dl, ts_dl, inbreast_dl, classes, cls_counts = get_dls(train_df,val_df, test_df, inbreast_df, bs =16)

Class counts: {np.int64(0): np.int64(4824), np.int64(1): np.int64(1674), np.int64(2): np.int64(229), np.int64(3): np.int64(247), np.int64(4): np.int64(83)}
Smoothed class weights: [0.259 0.439 1.187 1.143 1.972]


In [21]:
# finding columns distribution
finding_cols = ['has_neg_b1','has_neg_b2','has_mass','has_calc','has_structural','has_lymph']
print(train_df[finding_cols].sum())

has_neg_b1        4824
has_neg_b2        1666
has_mass           414
has_calc           146
has_structural      72
has_lymph           18
dtype: int64


In [22]:
# THE KEY — joint distribution: for each finding, how many samples per BI-RADS class
for f in finding_cols:
    print(f"\n--- {f} ---")
    print(train_df[train_df[f]==1]['label'].value_counts().sort_index())


--- has_neg_b1 ---
label
0    4824
Name: count, dtype: int64

--- has_neg_b2 ---
label
1    1666
Name: count, dtype: int64

--- has_mass ---
label
2    195
3    156
4     63
Name: count, dtype: int64

--- has_calc ---
label
2    24
3    78
4    44
Name: count, dtype: int64

--- has_structural ---
label
1     7
2    12
3    39
4    14
Name: count, dtype: int64

--- has_lymph ---
label
1    1
3    9
4    8
Name: count, dtype: int64


In [ ]:
import os
import gc
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

from tqdm import tqdm
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)


FINDING_COLS = [
    "has_neg_b1",
    "has_neg_b2",
    "has_mass",
    "has_calc",
    "has_structural",
    "has_lymph",
]

NUM_FINDINGS = 6
NUM_CLASSES = 5

BIRADS_CLASS_NAMES = [
    "BI-RADS 1",
    "BI-RADS 2",
    "BI-RADS 3",
    "BI-RADS 4",
    "BI-RADS 5",
]

BIRADS_CORAL_CLASS_WEIGHTS = [0.1, 0.2, 1.5, 1.5, 1.8]
BIRADS_CORAL_LEVEL_WEIGHTS = [1.0, 1.2, 1.5, 1.8]

BIRADS_PROTO_MOMENTUM = [0.99, 0.99, 0.97, 0.97, 0.95]
FINDING_PROTO_MOMENTUM = [0.99, 0.99, 0.97, 0.97, 0.93, 0.93]


FINDING_SAMPLE_WEIGHT = [0.1, 0.2, 1.2, 1.2, 1.5, 1.5]
BIRADS_SAMPLE_WEIGHT = [0.1, 0.2, 1.00, 1.20, 1.50]
PROTO_MIN_UPDATES = 5


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


def labels_to_levels(labels, num_classes=5):
    labels = labels.long()
    levels = []
    for k in range(num_classes - 1):
        levels.append((labels > k).float())
    return torch.stack(levels, dim=1)


def coral_logits_to_labels(logits):
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).sum(dim=1)
    return preds.long()


class WeightedCORALLoss(nn.Module):
    def __init__(
        self,
        num_classes=5,
        class_weights=None,
        underestimation_weight=4.0,
        level_weights=None,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.underestimation_weight = underestimation_weight

        if class_weights is None:
            class_weights = BIRADS_CORAL_CLASS_WEIGHTS
        if level_weights is None:
            level_weights = BIRADS_CORAL_LEVEL_WEIGHTS

        self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float32))
        self.register_buffer("level_weights", torch.tensor(level_weights, dtype=torch.float32))

    def forward(self, logits, labels):
        labels = labels.long()
        levels = labels_to_levels(labels, self.num_classes).to(logits.device)

        bce = F.binary_cross_entropy_with_logits(logits, levels, reduction="none")
        bce = bce * self.level_weights.view(1, -1)

        probs = torch.sigmoid(logits)
        under_mask = (levels == 1.0) & (probs < 0.5)
        under_scale = torch.ones_like(bce)
        under_scale[under_mask] = self.underestimation_weight
        bce = bce * under_scale

        per_sample = bce.mean(dim=1)
        sample_w = self.class_weights[labels]

        return (per_sample * sample_w).sum() / (sample_w.sum() + 1e-8)


class ProtoInfoNCELoss(nn.Module):
    def __init__(self, temperature=0.07, min_updates=PROTO_MIN_UPDATES):
        super().__init__()
        self.tau = temperature
        self.min_updates = min_updates

    def forward(self, q, prototypes, proto_updates, sample_weights_list, pos_indices):
        ready = proto_updates >= self.min_updates
        losses = []
        weights = []

        for i in range(q.shape[0]):
            pos_idx = int(pos_indices[i])
            if pos_idx < 0 or not ready[pos_idx]:
                continue

            qi = q[i]
            sims = []
            pos_local_idx = None

            for p in range(prototypes.shape[0]):
                if not ready[p]:
                    continue

                sim = torch.dot(qi, prototypes[p]) / self.tau

                if p == pos_idx:
                    pos_local_idx = len(sims)

                sims.append(sim)

            if pos_local_idx is None or len(sims) < 2:
                continue

            sims = torch.stack(sims)
            loss_i = -F.log_softmax(sims, dim=0)[pos_local_idx]

            losses.append(loss_i)
            weights.append(float(sample_weights_list[i]))

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device, dtype=q.dtype)

        losses = torch.stack(losses)
        weights = torch.tensor(weights, device=q.device, dtype=q.dtype)

        return (weights * losses).sum() / (weights.sum() + 1e-8)


class MultiLabelProtoLoss(nn.Module):
    def __init__(self, temperature=0.07, min_updates=PROTO_MIN_UPDATES):
        super().__init__()
        self.tau = temperature
        self.min_updates = min_updates

    def forward(self, q, prototypes, proto_updates, sample_weights_list, finding_vec):
        ready = proto_updates >= self.min_updates
        losses = []
        weights = []

        for i in range(q.shape[0]):
            active_findings = torch.where(finding_vec[i] > 0.5)[0]
            
            if len(active_findings) == 0:
                continue

            qi = q[i]
            
            for finding_idx in active_findings:
                finding_idx = finding_idx.item()
                
                if not ready[finding_idx]:
                    continue

                all_sims = []
                pos_local_idx = None

                for p in range(prototypes.shape[0]):
                    if not ready[p]:
                        continue

                    sim = torch.dot(qi, prototypes[p]) / self.tau

                    if p == finding_idx:
                        pos_local_idx = len(all_sims)

                    all_sims.append(sim)

                if pos_local_idx is None or len(all_sims) < 2:
                    continue

                all_sims = torch.stack(all_sims)
                log_softmax = F.log_softmax(all_sims, dim=0)
                loss_i = -log_softmax[pos_local_idx]

                losses.append(loss_i)
                weights.append(sample_weights_list[i])

        if len(losses) == 0:
            return torch.tensor(0.0, device=q.device, dtype=q.dtype)

        losses = torch.stack(losses)
        weights = torch.tensor(weights, device=q.device, dtype=q.dtype)

        return (weights * losses).sum() / (weights.sum() + 1e-8)


class PairwiseSeparationLoss(nn.Module):
    def __init__(self, margin=0.5, min_updates=10):
        super().__init__()
        self.margin = margin
        self.min_updates = min_updates

    def forward(
        self,
        proto_finding,
        proto_birads,
        proto_finding_updates=None,
        proto_birads_updates=None,
    ):
        loss = proto_finding.new_tensor(0.0)
        count = 0

        for j in [1, 2, 3, 4]:
            if proto_birads_updates is not None:
                if proto_birads_updates[0] < self.min_updates or proto_birads_updates[j] < self.min_updates:
                    continue

            sim = torch.dot(proto_birads[0], proto_birads[j])
            loss += F.relu(sim - self.margin)
            count += 1

        for j in [2, 3, 4]:
            if proto_birads_updates is not None:
                if proto_birads_updates[1] < self.min_updates or proto_birads_updates[j] < self.min_updates:
                    continue

            sim = torch.dot(proto_birads[1], proto_birads[j])
            loss += F.relu(sim - self.margin)
            count += 1

        for j in [1, 2, 3, 4, 5]:
            if proto_finding_updates is not None:
                if proto_finding_updates[0] < self.min_updates or proto_finding_updates[j] < self.min_updates:
                    continue

            sim = torch.dot(proto_finding[0], proto_finding[j])
            loss += F.relu(sim - self.margin)
            count += 1

        for j in [2, 3, 4, 5]:
            if proto_finding_updates is not None:
                if proto_finding_updates[1] < self.min_updates or proto_finding_updates[j] < self.min_updates:
                    continue

            sim = torch.dot(proto_finding[1], proto_finding[j])
            loss += F.relu(sim - self.margin)
            count += 1

        if count == 0:
            return proto_finding.new_tensor(0.0)

        return loss / count


class FindingBiradsProtoEncoder(nn.Module):
    def __init__(self, backbone_name, backbone, emb_dim=128, dropout=0.2):
        super().__init__()
        self.backbone_name = backbone_name.lower()

        if "efficientnet" in self.backbone_name:
            self.features = backbone.features
            self.avgpool = backbone.avgpool
            self.num_features = backbone.classifier[1].in_features
        elif "convnext" in self.backbone_name:
            self.features = backbone.features
            self.avgpool = backbone.avgpool
            self.num_features = backbone.classifier[2].in_features
        else:
            raise ValueError(f"Unsupported backbone: {backbone_name}")

        self.birads_head = nn.Sequential(
            nn.Linear(self.num_features, 256),
            nn.LayerNorm(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, NUM_CLASSES - 1),
        )

        self.proj_head_finding = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self.proj_head_birads = nn.Sequential(
            nn.Linear(self.num_features, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.ReLU(inplace=True),
            nn.Linear(self.num_features, emb_dim),
        )

        self._init_weights()

    def _init_weights(self):
        heads = [self.birads_head, self.proj_head_finding, self.proj_head_birads]

        for head in heads:
            for m in head.modules():
                if isinstance(m, nn.Linear):
                    nn.init.trunc_normal_(m.weight, std=0.02)
                    if m.bias is not None:
                        nn.init.zeros_(m.bias)
                elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
                    if hasattr(m, "weight") and m.weight is not None:
                        nn.init.ones_(m.weight)
                    if hasattr(m, "bias") and m.bias is not None:
                        nn.init.zeros_(m.bias)

    def _extract(self, x):
        f = self.features(x)
        f = self.avgpool(f)
        f = torch.flatten(f, 1)
        return f

    def forward(self, x, return_embeddings=False):
        feat = self._extract(x)
        birads_logits = self.birads_head(feat)

        if return_embeddings:
            emb_f = F.normalize(self.proj_head_finding(feat), dim=1)
            emb_b = F.normalize(self.proj_head_birads(feat), dim=1)
            return birads_logits, emb_f, emb_b

        return birads_logits


class DualProtoNet(nn.Module):
    def __init__(
        self,
        backbone_name,
        backbone_fn,
        backbone_weights,
        emb_dim=128,
        dropout=0.2,
        num_classes=NUM_CLASSES,
        num_findings=NUM_FINDINGS,
        temperature=0.12,
    ):
        super().__init__()
        self.num_classes = num_classes
        self.num_findings = num_findings
        self.temperature = temperature
        self.emb_dim = emb_dim

        backbone = backbone_fn(weights=backbone_weights)

        self.encoder = FindingBiradsProtoEncoder(
            backbone_name=backbone_name,
            backbone=backbone,
            emb_dim=emb_dim,
            dropout=dropout,
        )

        self.register_buffer(
            "proto_finding",
            F.normalize(torch.randn(num_findings, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_finding_updates",
            torch.zeros(num_findings, dtype=torch.long),
        )

        self.register_buffer(
            "proto_birads",
            F.normalize(torch.randn(num_classes, emb_dim), dim=-1),
        )
        self.register_buffer(
            "proto_birads_updates",
            torch.zeros(num_classes, dtype=torch.long),
        )

        self.register_buffer(
            "finding_momentum",
            torch.tensor(FINDING_PROTO_MOMENTUM, dtype=torch.float32),
        )
        self.register_buffer(
            "birads_momentum",
            torch.tensor(BIRADS_PROTO_MOMENTUM, dtype=torch.float32),
        )

    @torch.no_grad()
    def update_finding_prototypes(self, emb_f, finding_vec):
        for f in range(self.num_findings):
            mask = finding_vec[:, f] > 0.5

            if mask.sum() == 0:
                continue

            batch_proto = F.normalize(emb_f[mask].mean(dim=0), dim=0)
            m = self.finding_momentum[f].item()

            if self.proto_finding_updates[f] == 0:
                self.proto_finding[f] = batch_proto
            else:
                self.proto_finding[f] = F.normalize(
                    m * self.proto_finding[f] + (1.0 - m) * batch_proto,
                    dim=0,
                )

            self.proto_finding_updates[f] += 1

    @torch.no_grad()
    def update_birads_prototypes(self, emb_b, labels):
        labels = labels.long()

        for c in range(self.num_classes):
            mask = labels == c

            if mask.sum() == 0:
                continue

            batch_proto = F.normalize(emb_b[mask].mean(dim=0), dim=0)
            m = self.birads_momentum[c].item()

            if self.proto_birads_updates[c] == 0:
                self.proto_birads[c] = batch_proto
            else:
                self.proto_birads[c] = F.normalize(
                    m * self.proto_birads[c] + (1.0 - m) * batch_proto,
                    dim=0,
                )

            self.proto_birads_updates[c] += 1

    def forward_encoder(self, x, return_embeddings=False):
        return self.encoder(x, return_embeddings=return_embeddings)


@torch.no_grad()
def validate(model, loader, device, coral_loss_fn, class_names=None):
    class_names = class_names or BIRADS_CLASS_NAMES
    model.eval()

    total_loss = 0.0
    preds_all = []
    targets_all = []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_encoder(imgs, return_embeddings=False)
        loss = coral_loss_fn(logits, labels)

        total_loss += loss.item()
        preds = coral_logits_to_labels(logits)

        preds_all.extend(preds.cpu().numpy())
        targets_all.extend(labels.cpu().numpy())

    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro", zero_division=0)
    weighted_f1 = f1_score(targets_all, preds_all, average="weighted", zero_division=0)
    cm = confusion_matrix(targets_all, preds_all, labels=list(range(NUM_CLASSES)))

    print("\n--- Validation (BI-RADS CORAL) ---")
    print(cm)
    print(
        classification_report(
            targets_all,
            preds_all,
            labels=list(range(NUM_CLASSES)),
            target_names=class_names,
            zero_division=0,
        )
    )

    return {
        "loss": total_loss / max(len(loader), 1),
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


@torch.no_grad()
def test_model(model, loader, device, save_dir, name="Test", class_names=None):
    class_names = class_names or BIRADS_CLASS_NAMES
    model.eval()

    preds_all = []
    targets_all = []

    for batch in loader:
        imgs = batch["qry_im"].to(device, non_blocking=True)
        labels = batch["qry_gt"].to(device, non_blocking=True).long()

        logits = model.forward_encoder(imgs, return_embeddings=False)
        preds = coral_logits_to_labels(logits)

        preds_all.extend(preds.cpu().numpy())
        targets_all.extend(labels.cpu().numpy())

    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro", zero_division=0)
    weighted_f1 = f1_score(targets_all, preds_all, average="weighted", zero_division=0)
    cm = confusion_matrix(targets_all, preds_all, labels=list(range(NUM_CLASSES)))

    report = classification_report(
        targets_all,
        preds_all,
        labels=list(range(NUM_CLASSES)),
        target_names=class_names,
        zero_division=0,
    )

    print(f"\n{'=' * 70}")
    print(f"{name} | Acc={acc:.4f} | Macro-F1={macro_f1:.4f} | Weighted-F1={weighted_f1:.4f}")
    print(cm)
    print(report)
    print("=" * 70)

    os.makedirs(save_dir, exist_ok=True)
    out_path = os.path.join(save_dir, f"{name.lower().replace(' ', '_')}_birads_report.txt")

    with open(out_path, "w") as fh:
        fh.write(f"{name}\n")
        fh.write(f"Acc={acc:.4f} | Macro-F1={macro_f1:.4f} | Weighted-F1={weighted_f1:.4f}\n\n")
        fh.write(str(cm) + "\n\n")
        fh.write(report)

    return {
        "acc": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }


def train_model(
    model,
    train_loader,
    val_loader,
    device,
    lr_backbone=5e-5,
    lr_heads=5e-5,
    epochs=60,
    patience=15,
    save_path="best_birads_coral_dual_proto_hiersep.pth",
    w_finding=0.5,
    w_birads=0.5,
    w_sep=0.1,
    sep_margin=0.3,
    warmup_epochs=3,
    ramp_epochs=5,
    class_names=None,
):
    class_names = class_names or BIRADS_CLASS_NAMES
    os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
    log_path = save_path.replace(".pth", "_log.txt")

    coral_loss_fn = WeightedCORALLoss(
        num_classes=NUM_CLASSES,
        class_weights=BIRADS_CORAL_CLASS_WEIGHTS,
        underestimation_weight=5.0,
        level_weights=BIRADS_CORAL_LEVEL_WEIGHTS,
    ).to(device)

    proto_loss_fn_finding = MultiLabelProtoLoss(
        temperature=model.temperature,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    proto_loss_fn_birads = ProtoInfoNCELoss(
        temperature=model.temperature,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    sep_loss_fn = PairwiseSeparationLoss(
        margin=sep_margin,
        min_updates=PROTO_MIN_UPDATES,
    ).to(device)

    optimizer = AdamW(
        [
            {
                "params": list(model.encoder.features.parameters()) + list(model.encoder.avgpool.parameters()),
                "lr": lr_backbone,
                "weight_decay": 0.05,
            },
            {
                "params": model.encoder.birads_head.parameters(),
                "lr": lr_heads,
                "weight_decay": 0.05,
            },
            {
                "params": model.encoder.proj_head_finding.parameters(),
                "lr": lr_heads,
                "weight_decay": 0.05,
            },
            {
                "params": model.encoder.proj_head_birads.parameters(),
                "lr": lr_heads,
                "weight_decay": 0.05,
            },
        ]
    )

    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None

    best_macro_f1 = 0.0
    not_improved = 0

    for epoch in range(epochs):
        if epoch < warmup_epochs:
            pw_f = 0.0
            pw_b = 0.0
            pw_s = 0.0
        else:
            ramp = min(1.0, (epoch - warmup_epochs + 1) / max(ramp_epochs, 1))
            pw_f = w_finding * ramp
            pw_b = w_birads * ramp
            pw_s = w_sep * ramp

        model.train()

        e_coral = 0.0
        e_proto_f = 0.0
        e_proto_b = 0.0
        e_sep = 0.0

        preds_all = []
        targets_all = []

        pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}")

        for batch in pbar:
            imgs = batch["qry_im"].to(device, non_blocking=True)
            labels = batch["qry_gt"].to(device, non_blocking=True).long()
            finding_vec = batch["finding_vec"].to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type=device.type, enabled=(scaler is not None)):
                logits, emb_f, emb_b = model.forward_encoder(imgs, return_embeddings=True)

                coral_loss = coral_loss_fn(logits, labels)

                if pw_f > 0:
                    proto_loss_f = proto_loss_fn_finding(
                        q=emb_f,
                        prototypes=model.proto_finding,
                        proto_updates=model.proto_finding_updates,
                        sample_weights_list=[FINDING_SAMPLE_WEIGHT[i] if i < len(FINDING_SAMPLE_WEIGHT) else 1.0 
                                            for i in range(imgs.shape[0])],
                        finding_vec=finding_vec,
                    )
                else:
                    proto_loss_f = torch.tensor(0.0, device=device)

                if pw_b > 0:
                    birads_pos_indices = labels.detach().cpu().tolist()
                    birads_sw = [BIRADS_SAMPLE_WEIGHT[int(lb)] for lb in birads_pos_indices]

                    proto_loss_b = proto_loss_fn_birads(
                        q=emb_b,
                        prototypes=model.proto_birads,
                        proto_updates=model.proto_birads_updates,
                        sample_weights_list=birads_sw,
                        pos_indices=birads_pos_indices,
                    )
                else:
                    proto_loss_b = torch.tensor(0.0, device=device)

                if pw_s > 0:
                    sep_loss = sep_loss_fn(
                        proto_finding=model.proto_finding,
                        proto_birads=model.proto_birads,
                        proto_finding_updates=model.proto_finding_updates,
                        proto_birads_updates=model.proto_birads_updates,
                    )
                else:
                    sep_loss = torch.tensor(0.0, device=device)

                total_loss = (
                    coral_loss
                    + pw_f * proto_loss_f
                    + pw_b * proto_loss_b
                    + pw_s * sep_loss
                )

            if scaler is not None:
                scaler.scale(total_loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                total_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), 1.0)
                optimizer.step()

            with torch.no_grad():
                feat = model.encoder._extract(imgs)

                emb_f_fresh = F.normalize(
                    model.encoder.proj_head_finding(feat),
                    dim=1,
                )
                model.update_finding_prototypes(emb_f_fresh, finding_vec)

                emb_b_fresh = F.normalize(
                    model.encoder.proj_head_birads(feat),
                    dim=1,
                )
                model.update_birads_prototypes(emb_b_fresh, labels)

            preds = coral_logits_to_labels(logits)
            preds_all.extend(preds.detach().cpu().numpy())
            targets_all.extend(labels.detach().cpu().numpy())

            e_coral += coral_loss.item()
            e_proto_f += proto_loss_f.item()
            e_proto_b += proto_loss_b.item()
            e_sep += sep_loss.item()

            pbar.set_postfix(
                coral=f"{coral_loss.item():.3f}",
                pf=f"{proto_loss_f.item():.3f}",
                pb=f"{proto_loss_b.item():.3f}",
                sep=f"{sep_loss.item():.3f}",
                pwf=f"{pw_f:.2f}",
                pwb=f"{pw_b:.2f}",
                pws=f"{pw_s:.2f}",
            )

        n = max(len(train_loader), 1)

        train_acc = accuracy_score(targets_all, preds_all)
        train_macro_f1 = f1_score(targets_all, preds_all, average="macro", zero_division=0)
        train_wt_f1 = f1_score(targets_all, preds_all, average="weighted", zero_division=0)

        print(f"\n{'=' * 70}")
        print(
            f"Epoch {epoch + 1}/{epochs} | "
            f"coral={e_coral / n:.4f} | "
            f"proto_f={e_proto_f / n:.4f} | "
            f"proto_b={e_proto_b / n:.4f} | "
            f"sep={e_sep / n:.4f} | "
            f"pwf={pw_f:.3f} | "
            f"pwb={pw_b:.3f} | "
            f"pws={pw_s:.3f} | "
            f"train_acc={train_acc:.4f} | "
            f"train_macro_f1={train_macro_f1:.4f}"
        )

        print(
            f"Proto finding updates: {model.proto_finding_updates.cpu().tolist()} | "
            f"Proto birads updates: {model.proto_birads_updates.cpu().tolist()}"
        )

        print("\n--- Train (BI-RADS CORAL) ---")
        print(
            classification_report(
                targets_all,
                preds_all,
                labels=list(range(NUM_CLASSES)),
                target_names=class_names,
                zero_division=0,
            )
        )

        val_metrics = validate(
            model=model,
            loader=val_loader,
            device=device,
            coral_loss_fn=coral_loss_fn,
            class_names=class_names,
        )

        print(
            f"Val loss={val_metrics['loss']:.4f} | "
            f"Val acc={val_metrics['acc']:.4f} | "
            f"Val macro_f1={val_metrics['macro_f1']:.4f} | "
            f"Val weighted_f1={val_metrics['weighted_f1']:.4f}"
        )
        print("=" * 70)

        scheduler.step()

        with open(log_path, "a") as fh:
            fh.write(
                f"Epoch {epoch + 1} | "
                f"coral={e_coral / n:.4f} | "
                f"proto_f={e_proto_f / n:.4f} | "
                f"proto_b={e_proto_b / n:.4f} | "
                f"sep={e_sep / n:.4f} | "
                f"pwf={pw_f:.3f} | "
                f"pwb={pw_b:.3f} | "
                f"pws={pw_s:.3f} | "
                f"train_acc={train_acc:.4f} | "
                f"train_macro_f1={train_macro_f1:.4f} | "
                f"train_wt_f1={train_wt_f1:.4f} | "
                f"val_loss={val_metrics['loss']:.4f} | "
                f"val_acc={val_metrics['acc']:.4f} | "
                f"val_macro_f1={val_metrics['macro_f1']:.4f} | "
                f"val_wt_f1={val_metrics['weighted_f1']:.4f}\n"
            )

        if val_metrics["macro_f1"] > best_macro_f1:
            best_macro_f1 = val_metrics["macro_f1"]

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "best_macro_f1": best_macro_f1,
                    "config": {
                        "num_classes": NUM_CLASSES,
                        "num_findings": NUM_FINDINGS,
                        "w_finding": w_finding,
                        "w_birads": w_birads,
                        "w_sep": w_sep,
                        "sep_margin": sep_margin,
                        "temperature": model.temperature,
                        "class_names": class_names,
                    },
                },
                save_path,
            )

            print(f"Saved best model macro-F1={best_macro_f1:.4f}")
            not_improved = 0
        else:
            not_improved += 1

            if not_improved >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    return best_macro_f1


def run_experiments(train_loader, val_loader, test_loader, inbreast_loader):
    seed_everything(42)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    train_config = dict(
        lr_backbone=5e-5,
        lr_heads=5e-5,
        epochs=60,
        patience=15,
        w_finding=0.5,
        w_birads=0.5,
        w_sep=0.5,
        sep_margin=0.25,
        warmup_epochs=3,
        ramp_epochs=5,
        class_names=BIRADS_CLASS_NAMES,
    )

    backbones = [

        {
            "name": "convnext_base",
            "fn": models.convnext_base,
            "w": models.ConvNeXt_Base_Weights.DEFAULT,
        },
        {
            "name": "efficientnet_b3",
            "fn": models.efficientnet_b3,
            "w": models.EfficientNet_B3_Weights.DEFAULT,
        },

    ]

    all_results = {}

    for cfg in backbones:
        out_dir = f"/workspace/Thesis_updated_results/birads_512_dual_proto/{cfg['name']}"
        save_path = f"{out_dir}/best_model.pth"

        os.makedirs(out_dir, exist_ok=True)

        print(f"\n{'#' * 70}")
        print(f"Backbone: {cfg['name']}")
        print("Task: BI-RADS CORAL + Finding Proto + BI-RADS Proto + Hierarchical Separation")
        print(f"CORAL class weights: {BIRADS_CORAL_CLASS_WEIGHTS}")
        print(f"CORAL level weights: {BIRADS_CORAL_LEVEL_WEIGHTS}")
        print(f"Finding proto momentums: {FINDING_PROTO_MOMENTUM}")
        print(f"BI-RADS proto momentums: {BIRADS_PROTO_MOMENTUM}")
        print(f"Separation weight: {train_config['w_sep']}")
        print(f"Separation margin: {train_config['sep_margin']}")
        print(f"{'#' * 70}")

        model = DualProtoNet(
            backbone_name=cfg["name"],
            backbone_fn=cfg["fn"],
            backbone_weights=cfg["w"],
            emb_dim=128,
            dropout=0.2,
            num_classes=NUM_CLASSES,
            num_findings=NUM_FINDINGS,
            temperature=0.07,
        ).to(device)

        num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Trainable params: {num_params:,}")

        best_macro_f1 = train_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            save_path=save_path,
            **train_config,
        )

        ckpt = torch.load(save_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])

        print(f"Loaded epoch {ckpt['epoch'] + 1} | best macro-F1={ckpt['best_macro_f1']:.4f}")

        vindr_metrics = test_model(
            model=model,
            loader=test_loader,
            device=device,
            save_dir=out_dir,
            name="VinDr-Test",
            class_names=BIRADS_CLASS_NAMES,
        )

        inbreast_metrics = test_model(
            model=model,
            loader=inbreast_loader,
            device=device,
            save_dir=out_dir,
            name="INbreast",
            class_names=BIRADS_CLASS_NAMES,
        )

        all_results[cfg["name"]] = {
            "best_val_macro_f1": best_macro_f1,
            "vindr_acc": vindr_metrics["acc"],
            "vindr_macro_f1": vindr_metrics["macro_f1"],
            "vindr_weighted_f1": vindr_metrics["weighted_f1"],
            "inbreast_acc": inbreast_metrics["acc"],
            "inbreast_macro_f1": inbreast_metrics["macro_f1"],
            "inbreast_weighted_f1": inbreast_metrics["weighted_f1"],
        }

        del model, ckpt
        torch.cuda.empty_cache()
        gc.collect()

    print(f"\n{'=' * 95}")
    print("FINAL BI-RADS CORAL + DUAL PROTO + HIERARCHICAL SEPARATION RESULTS")
    print(f"{'=' * 95}")
    print(
        f"{'Model':<22} "
        f"{'Val Macro-F1':>14} "
        f"{'VinDr Macro-F1':>17} "
        f"{'INbreast Macro-F1':>20}"
    )
    print("-" * 95)

    for name, r in all_results.items():
        print(
            f"{name:<22} "
            f"{r['best_val_macro_f1']:>14.4f} "
            f"{r['vindr_macro_f1']:>17.4f} "
            f"{r['inbreast_macro_f1']:>20.4f}"
        )

    return all_results


if __name__ == "__main__":
    results = run_experiments(
        train_loader=tr_dl,
        val_loader=val_dl,
        test_loader=ts_dl,
        inbreast_loader=inbreast_dl,
    )

Device: cuda

######################################################################
Backbone: convnext_base
Task: BI-RADS CORAL + Finding Proto + BI-RADS Proto + Hierarchical Separation
CORAL class weights: [0.1, 0.2, 1.5, 1.5, 1.8]
CORAL level weights: [1.0, 1.2, 1.5, 1.8]
Finding proto momentums: [0.99, 0.99, 0.97, 0.97, 0.93, 0.93]
BI-RADS proto momentums: [0.99, 0.99, 0.97, 0.97, 0.95]
Separation weight: 0.5
Separation margin: 0.25
######################################################################
Trainable params: 90,194,052


Epoch 1/60: 100%|██████████| 441/441 [07:08<00:00,  1.03it/s, coral=1.044, pb=0.000, pf=0.000, pwb=0.00, pwf=0.00, pws=0.00, sep=0.000]


Epoch 1/60 | coral=1.5267 | proto_f=0.0000 | proto_b=0.0000 | sep=0.0000 | pwf=0.000 | pwb=0.000 | pws=0.000 | train_acc=0.1151 | train_macro_f1=0.0782
Proto finding updates: [441, 437, 278, 127, 69, 18] | Proto birads updates: [441, 437, 182, 181, 79]

--- Train (BI-RADS CORAL) ---
              precision    recall  f1-score   support

   BI-RADS 1       0.62      0.01      0.02      4823
   BI-RADS 2       0.25      0.38      0.30      1674
   BI-RADS 3       0.03      0.55      0.06       229
   BI-RADS 4       0.02      0.01      0.01       247
   BI-RADS 5       0.00      0.00      0.00        83

    accuracy                           0.12      7056
   macro avg       0.18      0.19      0.08      7056
weighted avg       0.49      0.12      0.09      7056




--- Validation (BI-RADS CORAL) ---
[[  0 182 356   0   0]
 [  0  84 112   0   0]
 [  0   6  17   0   0]
 [  0   9  14   0   0]
 [  0   4   3   0   0]]
              precision    recall  f1-score   support

   BI-RADS 1       0.00      0.00      0.00       538
   BI-RADS 2       0.29      0.43      0.35       196
   BI-RADS 3       0.03      0.74      0.06        23
   BI-RADS 4       0.00      0.00      0.00        23
   BI-RADS 5       0.00      0.00      0.00         7

    accuracy                           0.13       787
   macro avg       0.07      0.23      0.08       787
weighted avg       0.07      0.13      0.09       787

Val loss=1.3379 | Val acc=0.1283 | Val macro_f1=0.0828 | Val weighted_f1=0.0889
Saved best model macro-F1=0.0828


Epoch 2/60: 100%|██████████| 441/441 [06:25<00:00,  1.14it/s, coral=2.987, pb=0.000, pf=0.000, pwb=0.00, pwf=0.00, pws=0.00, sep=0.000]


...


Epoch 9/60 | coral=0.8578 | proto_f=0.6591 | proto_b=1.0266 | sep=0.4471 | pwf=0.500 | pwb=0.500 | pws=0.500 | train_acc=0.4734 | train_macro_f1=0.4483
Proto finding updates: [3969, 3922, 2459, 1131, 602, 157] | Proto birads updates: [3969, 3923, 1646, 1715, 697]

--- Train (BI-RADS CORAL) ---
              precision    recall  f1-score   support

   BI-RADS 1       0.91      0.42      0.57      4823
   BI-RADS 2       0.30      0.64      0.41      1674
   BI-RADS 3       0.12      0.44      0.18       229
   BI-RADS 4       0.37      0.36      0.36       247
   BI-RADS 5       0.70      0.75      0.72        83

    accuracy                           0.47      7056
   macro avg       0.48      0.52      0.45      7056
weighted avg       0.72      0.47      0.51      7056


--- Validation (BI-RADS CORAL) ---
[[347 138  46   5   2]
 [ 32 100  55   9   0]
 [  5   3  10   5   0]
 [  2   3   5   9   4]
 [  0   0   1   0   6]]
              precision    recall  f1-score   support

   BI-RA

Epoch 10/60: 100%|██████████| 441/441 [04:31<00:00,  1.63it/s, coral=0.380, pb=0.523, pf=0.333, pwb=0.50, pwf=0.50, pws=0.50, sep=0.417]



Epoch 10/60 | coral=0.8227 | proto_f=0.6343 | proto_b=0.9927 | sep=0.4400 | pwf=0.500 | pwb=0.500 | pws=0.500 | train_acc=0.5289 | train_macro_f1=0.4895
Proto finding updates: [4410, 4357, 2733, 1256, 671, 174] | Proto birads updates: [4410, 4358, 1827, 1923, 769]

--- Train (BI-RADS CORAL) ---
              precision    recall  f1-score   support

   BI-RADS 1       0.91      0.49      0.63      4824
   BI-RADS 2       0.32      0.66      0.44      1674
   BI-RADS 3       0.15      0.48      0.23       229
   BI-RADS 4       0.42      0.41      0.41       247
   BI-RADS 5       0.70      0.77      0.73        82

    accuracy                           0.53      7056
   macro avg       0.50      0.56      0.49      7056
weighted avg       0.73      0.53      0.57      7056


--- Validation (BI-RADS CORAL) ---
[[357 163  13   4   1]
 [ 32 136  22   6   0]
 [  5   9   5   4   0]
 [  2   4   7   6   4]
 [  0   0   1   0   6]]
              precision    recall  f1-score   support

   BI-R

Epoch 11/60: 100%|██████████| 441/441 [04:37<00:00,  1.59it/s, coral=0.431, pb=0.722, pf=0.246, pwb=0.50, pwf=0.50, pws=0.50, sep=0.434]



Epoch 11/60 | coral=0.8211 | proto_f=0.6286 | proto_b=0.9943 | sep=0.4128 | pwf=0.500 | pwb=0.500 | pws=0.500 | train_acc=0.5914 | train_macro_f1=0.5226
Proto finding updates: [4851, 4787, 3010, 1386, 738, 192] | Proto birads updates: [4851, 4788, 2011, 2105, 845]

--- Train (BI-RADS CORAL) ---
              precision    recall  f1-score   support

   BI-RADS 1       0.91      0.58      0.71      4823
   BI-RADS 2       0.36      0.64      0.46      1674
   BI-RADS 3       0.17      0.53      0.26       229
   BI-RADS 4       0.51      0.40      0.45       247
   BI-RADS 5       0.69      0.78      0.73        83

    accuracy                           0.59      7056
   macro avg       0.53      0.59      0.52      7056
weighted avg       0.74      0.59      0.63      7056

